# MetroBus Analytics
## Fase 1 - Exploración y Análisis de Datos (EDA)

### Autor
Manuel Mundo Expert

### Descripción del proyecto

MetroBus es una operadora pública de transporte urbano que busca mejorar la toma de decisiones mediante el análisis de datos operativos.

En esta fase se realizará un análisis exploratorio para comprender la estructura de los datos, evaluar su calidad e identificar posibles problemas antes de iniciar la fase de limpieza y transformación.
### Dataset disponible

El proyecto incluye 9 archivos CSV organizados en:

#### Tablas de hechos
- fact_viajes
- fact_incidencias
- fact_mantenimiento

#### Tablas de dimensiones
- dim_linea
- dim_vehiculo
- dim_conductor
- dim_parada
- dim_tarifa
- dim_depot
### Objetivos

- Revisar la estructura de las tablas.
- Analizar valores nulos.
- Detectar registros duplicados.
- Identificar outliers.
- Realizar visualizaciones exploratorias.
- Obtener conclusiones iniciales.

In [ ]:
# Imports básicos para análisis y visualización
import sys
import os
import math
import datetime as dt

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams

import seaborn as sns
from pathlib import Path
import warnings

# Configuración visual por defecto
sns.set(style="whitegrid", context="notebook", palette="deep")
rcParams['figure.figsize'] = (10, 6)
rcParams['axes.titlesize'] = 14
rcParams['axes.labelsize'] = 12
rcParams['legend.fontsize'] = 11
rcParams['xtick.labelsize'] = 11
rcParams['ytick.labelsize'] = 11

# Mostrar más columnas en pantalla
pd.options.display.max_columns = 200

# Semilla para reproducibilidad
RANDOM_SEED = 42

warnings.filterwarnings("ignore")

print("Librerías cargadas correctamente")
print("pandas:", pd.__version__, "| numpy:", np.__version__,
      "| matplotlib:", matplotlib.__version__, "| seaborn:", sns.__version__)


# 1. Carga de datos

En esta sección se cargan los nueve archivos CSV proporcionados por MetroBus y se verifica que la lectura de los datos se realiza correctamente.

In [10]:
import os

csv_files = [f for f in os.listdir() if f.endswith('.csv')]

print("Archivos CSV encontrados:")
for f in csv_files:
    print("-", f)

Archivos CSV encontrados:
- fact_viajes.csv


In [12]:
ruta = r"C:\Users\Usuario\Desktop\Master_PowerBi_Mibus\metrobus_dataset"

fact_viajes = pd.read_csv(f"{ruta}\\fact_viajes.csv")
fact_incidencias = pd.read_csv(f"{ruta}\\fact_incidencias.csv")
fact_mantenimiento = pd.read_csv(f"{ruta}\\fact_mantenimiento.csv")

dim_linea = pd.read_csv(f"{ruta}\\dim_linea.csv")
dim_vehiculo = pd.read_csv(f"{ruta}\\dim_vehiculo.csv")
dim_conductor = pd.read_csv(f"{ruta}\\dim_conductor.csv")
dim_parada = pd.read_csv(f"{ruta}\\dim_parada.csv")
dim_tarifa = pd.read_csv(f"{ruta}\\dim_tarifa.csv")
dim_depot = pd.read_csv(f"{ruta}\\dim_depot.csv")

print("Todos los archivos cargados correctamente")


Todos los archivos cargados correctamente


# 2. Revisión inicial de las tablas

Una vez cargados los datos, se revisan las dimensiones, los tipos de datos y una muestra inicial de registros para comprender la estructura del dataset.

In [13]:
tablas = {
    "fact_viajes": fact_viajes,
    "fact_incidencias": fact_incidencias,
    "fact_mantenimiento": fact_mantenimiento,
    "dim_linea": dim_linea,
    "dim_vehiculo": dim_vehiculo,
    "dim_conductor": dim_conductor,
    "dim_parada": dim_parada,
    "dim_tarifa": dim_tarifa,
    "dim_depot": dim_depot
}

In [14]:
for nombre, df in tablas.items():

    print("=" * 60)
    print(f"TABLA: {nombre}")
    print("=" * 60)

    print(f"Dimensiones: {df.shape}")

    display(df.head())

    print("\nTipos de datos:")
    print(df.dtypes)

    print("\n")

TABLA: fact_viajes
Dimensiones: (50000, 24)


,viaje_id,linea_id,vehiculo_id,conductor_id,parada_origen_id,parada_destino_id,fecha,anno,mes,dia_semana,es_festivo,franja_horaria,hora_salida_prog,hora_salida_real,hora_llegada_real,retraso_salida_min,duracion_real_min,pasajeros_subidos,ocupacion_pct,km_programados,km_recorridos,viaje_completado,consumo,tarifa_predominante_id
0,1,4,2,2,17,116,2023-08-14,2023,8,Monday,False,Tarde punta,17:50,18:09,19:22,19,73,78.0,0.609,14.1,14.1,True,4.69,4
1,2,4,3,3,5,48,2024-05-20,2024,5,Monday,False,Manana punta,07:10,07:23,08:30,13,67,115.0,0.885,14.1,14.1,True,5.86,2
2,3,5,4,4,33,110,2023-08-15,2023,8,Tuesday,False,Valle manana,12:30,12:30,13:05,0,35,47.0,0.362,7.3,7.3,True,3.15,3
3,4,2,5,5,37,116,2022-03-23,2022,3,Wednesday,False,Tarde punta,20:50,20:50,21:36,0,46,111.0,0.888,11.2,11.2,True,3.50,9
4,5,5,6,6,12,115,2022-03-24,2022,3,Thursday,False,Tarde punta,18:10,18:18,18:50,8,32,75.0,0.405,7.3,7.3,True,3.13,3



Tipos de datos:
viaje_id                    int64
linea_id                    int64
vehiculo_id                 int64
conductor_id                int64
parada_origen_id            int64
parada_destino_id           int64
fecha                      object
anno                        int64
mes                         int64
dia_semana                 object
es_festivo                   bool
franja_horaria             object
hora_salida_prog           object
hora_salida_real           object
hora_llegada_real          object
retraso_salida_min          int64
duracion_real_min           int64
pasajeros_subidos         float64
ocupacion_pct             float64
km_programados            float64
km_recorridos             float64
viaje_completado             bool
consumo                   float64
tarifa_predominante_id      int64
dtype: object


TABLA: fact_incidencias
Dimensiones: (4000, 16)


,incidencia_id,viaje_id,vehiculo_id,conductor_id,linea_id,fecha,anno,mes,hora_incidencia,tipo_incidencia,categoria,severidad,requiere_retirada,duracion_resolucion_min,vehiculo_sustituto,coste_estimado_eur
0,1,33554,39,11,4,2022-06-01,2022,6,19:11,Huelga parcial,Operacional,Alta,True,60,False,0.00
1,2,9428,21,21,6,2023-02-16,2023,2,20:42,Fallo electrico,Vehiculo,Alta,True,62,False,575.51
2,3,200,33,5,8,2022-01-20,2022,1,06:21,Accidente leve,Seguridad,Alta,True,29,True,3318.00
3,4,12448,17,17,10,2023-01-16,2023,1,02:30,Vandalismo,Seguridad,Media,False,16,False,651.99
4,5,39490,11,11,6,2023-01-08,2023,1,14:51,Retencion trafico,Externo,Baja,False,14,False,49.88



Tipos de datos:
incidencia_id                int64
viaje_id                     int64
vehiculo_id                  int64
conductor_id                 int64
linea_id                     int64
fecha                       object
anno                         int64
mes                          int64
hora_incidencia             object
tipo_incidencia             object
categoria                   object
severidad                   object
requiere_retirada             bool
duracion_resolucion_min      int64
vehiculo_sustituto            bool
coste_estimado_eur         float64
dtype: object


TABLA: fact_mantenimiento
Dimensiones: (876, 15)


...


Tipos de datos:
parada_id              int64
nombre_parada         object
barrio                object
tipo                  object
latitud              float64
longitud             float64
accesible_silla       object
marquesina              bool
panel_informacion       bool
activa                  bool
dtype: object


TABLA: dim_tarifa
Dimensiones: (9, 6)


,tarifa_id,tipo_titulo,categoria,precio_eur,es_abono,bonificado
0,1,Ordinario,Adulto,1.50,False,False
1,2,Bono 10 Viajes,Adulto,0.97,True,False
2,3,Abono Mensual,Adulto,0.60,True,False
3,4,Abono Joven,Joven,0.40,True,True
4,5,Abono Jubilado,Jubilado,0.25,True,True



Tipos de datos:
tarifa_id        int64
tipo_titulo     object
categoria       object
precio_eur     float64
es_abono          bool
bonificado        bool
dtype: object


TABLA: dim_depot
Dimensiones: (3, 6)


,depot_id,nombre,barrio,latitud,longitud,capacidad_vehiculos
0,1,Cochera Norte,Barrio Norte,40.440,-3.720,20
1,2,Cochera Central,Centro,40.418,-3.700,15
2,3,Cochera Sur,Barrio Sur,40.395,-3.695,12



Tipos de datos:
depot_id                 int64
nombre                  object
barrio                  object
latitud                float64
longitud               float64
capacidad_vehiculos      int64
dtype: object




## Resumen de dimensiones

A continuación se muestra un resumen del tamaño de cada tabla del dataset.

In [16]:
resumen = []

for nombre, df in tablas.items():
    resumen.append({
        "Tabla": nombre,
        "Filas": df.shape[0],
        "Columnas": df.shape[1]
    })

resumen_df = pd.DataFrame(resumen)

resumen_df

,Tabla,Filas,Columnas
0,fact_viajes,50000,24
1,fact_incidencias,4000,16
2,fact_mantenimiento,876,15
3,dim_linea,10,7
4,dim_vehiculo,45,12
5,dim_conductor,30,10
6,dim_parada,120,10
7,dim_tarifa,9,6
8,dim_depot,3,6


## Observaciones iniciales

Tras la revisión de la estructura de los datos se identifican los siguientes aspectos:

- El dataset está organizado siguiendo un modelo dimensional compuesto por tablas de hechos y dimensiones.
- La tabla principal es `fact_viajes`, con 50.000 registros, representando cada expedición realizada por MetroBus.
- Las tablas `fact_incidencias` y `fact_mantenimiento` complementan la información operativa con eventos relacionados con incidencias y mantenimiento.
- Las dimensiones contienen información descriptiva sobre líneas, vehículos, conductores, paradas, tarifas y cocheras.
- El tamaño de las dimensiones es consistente con el contexto de negocio descrito por MetroBus (10 líneas, 45 vehículos y 3 cocheras).

A continuación se realizará un análisis de calidad de datos para identificar posibles valores nulos, registros duplicados e inconsistencias.